In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from scipy.stats import chi2_contingency
import warnings
import os

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

# ── Paths ────────────────────────────────────────────────────────────────────
OUTPUT = '../UK_Accident_Analyzed/'
os.makedirs(OUTPUT, exist_ok=True)

def save(fig, name):
    """Save a matplotlib figure to the OUTPUT directory and close it."""
    fig.savefig(f'{OUTPUT}{name}.png', bbox_inches='tight', dpi=120)
    plt.close(fig)

# ── Load cleaned data ────────────────────────────────────────────────────────
df = pd.read_csv('../UK_Accident_cleaned.csv', low_memory=False)
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

# ── Recreate derived columns (originally built during EDA) ────────────────────
df['Road_Type_Label'] = df['Road_Type'].fillna('Unknown')
df['Is_Fatal'] = (df['Accident_Severity'] == 1).astype(int)

df['Light_Simple'] = df['Light_Conditions'].map({
    'Daylight: Street light present'           : 'Daylight',
    'Darkness: Street lights present and lit'  : 'Dark-lit',
    'Darkness: Street lights present but unlit': 'Dark-unlit',
    'Darkeness: No street lighting'            : 'Dark-no light',
    'Darkness: Street lighting unknown'        : 'Dark-unknown'
})

df['Weather_Simple'] = df['Weather_Conditions'].map({
    'Fine without high winds' : 'Fine',
    'Fine with high winds'    : 'Fine+Wind',
    'Raining without high winds': 'Rain',
    'Raining with high winds' : 'Rain+Wind',
    'Snowing without high winds': 'Snow',
    'Snowing with high winds' : 'Snow+Wind',
    'Fog or mist'             : 'Fog',
    'Other'                   : 'Other',
    'Unknown'                 : 'Unknown'
})

df['Surface_Simple'] = df['Road_Surface_Conditions'].map({
    'Dry': 'Dry', 'Normal': 'Normal', 'Wet/Damp': 'Wet',
    'Snow': 'Snow', 'Frost/Ice': 'Ice', 'Flood (Over 3cm of water)': 'Flood'
})

df['Time_Period'] = pd.cut(
    df['Hour'],
    bins=[-1, 6, 9, 12, 16, 19, 22, 24],
    labels=['Night(0-6)', 'AM Rush(7-9)', 'Midday(10-12)',
            'Afternoon(13-16)', 'PM Rush(17-19)', 'Evening(20-22)', 'Late Night(23-24)']
)

df['Day_Name'] = df['Day_of_Week'].map(
    {1:'Sunday',2:'Monday',3:'Tuesday',4:'Wednesday',5:'Thursday',6:'Friday',7:'Saturday'}
) if 'Day_Name' not in df.columns else df['Day_Name']

df['Day_Type'] = df['Day_Name'].apply(lambda x: 'Weekend' if x in ['Saturday', 'Sunday'] else 'Weekday')
df['Vehicle_Count_Type'] = df['Number_of_Vehicles'].apply(lambda x: 'Single' if x == 1 else 'Multi')

# ── Pre-compute variables used in final summary ──────────────────────────────
hour_fatal_rate = df.groupby('Hour')['Accident_Severity'].agg(
    total='count', fatal=lambda x: (x == 1).sum()
)
hour_fatal_rate['fatal_rate'] = (hour_fatal_rate['fatal'] / hour_fatal_rate['total'] * 100).round(2)

hour_counts = df.groupby('Hour').size()

print(f'Loaded cleaned data: {df.shape}')

---
## Step 7 — Correlation & Feature Importance

Two complementary approaches:
1. **Pearson correlation matrix** — linear relationships between numeric variables
2. **Random Forest feature importance** — non-linear importance ranking (handles categorical variables after encoding)
3. **Chi-square tests** — statistical association between each categorical predictor and accident severity

In [86]:
# ── 7a. Pearson correlation matrix ────────────────────────────────────────────
num_cols = [
    'Accident_Severity', 'Number_of_Vehicles', 'Number_of_Casualties',
    'Speed_limit', 'Hour', 'Month', 'Day_of_Week', 'Urban_or_Rural_Area'
]
corr_df     = df[num_cols].dropna()
corr_matrix = corr_df.corr()
print(corr_matrix['Accident_Severity'].sort_values())

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax, linewidths=0.5)
ax.set_title('Correlation Matrix', fontsize=14, fontweight='bold')
save(fig, '25_correlation_matrix')

Number_of_Casualties   -0.079078
Urban_or_Rural_Area    -0.074161
Speed_limit            -0.071081
Month                  -0.001082
Day_of_Week             0.002566
Hour                    0.004318
Number_of_Vehicles      0.080853
Accident_Severity       1.000000
Name: Accident_Severity, dtype: float64


In [87]:
# ── 7b. Random Forest Feature Importance ──────────────────────────────────────
# String columns are label-encoded (ordinal assumption is a simplification but
# acceptable for importance ranking purposes — not prediction).
# Data is downsampled to 200k rows so training completes in reasonable time.
feature_cols = [
    'Speed_limit', 'Hour', 'Day_of_Week', 'Month', 'Urban_or_Rural_Area',
    'Number_of_Vehicles', 'Road_Type', 'Light_Conditions',
    'Weather_Conditions', 'Road_Surface_Conditions'
]

rf_df    = df[feature_cols + ['Accident_Severity']].dropna().copy()
str_cols = ['Road_Type', 'Light_Conditions', 'Weather_Conditions', 'Road_Surface_Conditions']
for col in str_cols:
    le = LabelEncoder()
    rf_df[col] = le.fit_transform(rf_df[col].astype(str))

rf_sample = rf_df.sample(min(200_000, len(rf_df)), random_state=42)
X = rf_sample[feature_cols]
y = rf_sample['Accident_Severity']

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, max_depth=8)
rf.fit(X, y)

importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(importances)

fig, ax = plt.subplots(figsize=(10, 6))
importances.plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
ax.set_title('Feature Importance for Accident Severity (Random Forest)', fontsize=14, fontweight='bold')
ax.set_ylabel('Importance Score')
plt.xticks(rotation=30, ha='right')
save(fig, '26_feature_importance')

Number_of_Vehicles         0.447496
Hour                       0.105904
Speed_limit                0.096622
Road_Type                  0.069729
Urban_or_Rural_Area        0.065030
Light_Conditions           0.058309
Month                      0.042261
Weather_Conditions         0.041501
Day_of_Week                0.038340
Road_Surface_Conditions    0.034809
dtype: float64


In [88]:
# ── 7c. Chi-square tests — statistical significance of each predictor ─────────
# H0: no association between predictor and severity
# Low p-value → reject H0 → significant association
chi_cols = [
    'Speed_limit', 'Light_Conditions', 'Weather_Conditions',
    'Road_Surface_Conditions', 'Road_Type', 'Urban_or_Rural_Area', 'Day_of_Week'
]
chi_results = []
for col in chi_cols:
    sub  = df[[col, 'Accident_Severity']].dropna().reset_index(drop=True)
    ct   = pd.crosstab(sub[col], sub['Accident_Severity'])
    chi2, p, dof, _ = chi2_contingency(ct)
    chi_results.append({'feature': col, 'chi2': round(chi2, 2), 'p_value': p, 'significant': p < 0.05})
chi_df = pd.DataFrame(chi_results).sort_values('chi2', ascending=False)
print(chi_df)

                   feature     chi2        p_value  significant
0              Speed_limit  8945.39   0.000000e+00         True
5      Urban_or_Rural_Area  5951.40   0.000000e+00         True
1         Light_Conditions  5532.46   0.000000e+00         True
4                Road_Type  2533.34   0.000000e+00         True
6              Day_of_Week  1346.73  4.226763e-281         True
2       Weather_Conditions   660.17  8.096760e-132         True
3  Road_Surface_Conditions   276.01   1.810183e-53         True


---
## Step 8 — Geographic Analysis

Identify which **local authority districts** and **police force areas** have the highest accident and fatality counts.  
Also compare average casualties between urban and rural areas.

In [89]:
# ── 8a. Top 15 districts by total accidents ────────────────────────────────────
top_districts = df.groupby('Local_Authority_(District)').size().nlargest(15)
print(top_districts)

# ── 8b. Top 15 districts by fatal accident count ──────────────────────────────
district_fatal2 = df[df['Accident_Severity'] == 1].groupby('Local_Authority_(District)').size().nlargest(15)
print(district_fatal2)

# ── 8c. Top police forces by fatal accident count ─────────────────────────────
force_fatal = df[df['Accident_Severity'] == 1].groupby('Police_Force').size().nlargest(10)
print(force_fatal)

# ── 8d. Average casualties: urban vs rural ────────────────────────────────────
print(df.groupby('Area_Type')['Number_of_Casualties'].mean().round(3))

Local_Authority_(District)
300    27675
102    13866
1      13559
926    13355
91     12452
215    12347
9      10086
30      9724
8       8791
20      8667
346     8658
211     8596
27      8523
10      7980
5       7944
dtype: int64

Local_Authority_(District)
300    235
927    181
231    167
753    139
596    132
102    127
926    124
169    107
938    107
211    103
351    102
479     98
285     96
91      95
286     95
dtype: int64

Police_Force
1     1465
43     805
97     600
6      582
20     576
50     564
42     510
22     504
4      440
32     393
dtype: int64

Area_Type
Rural          1.489
Unallocated    1.347
Urban          1.279
Name: Number_of_Casualties, dtype: float64


---
## Step 9 — Deep Dives: High-Risk Scenario Profiling

Drill into specific compound scenarios identified by earlier analysis:  
- Rural + high speed (60/70 mph)  
- Night-time + high speed  
- Single carriageway + rural  
- Urban vs Rural fatal rate trend over years  
- Single vs multi-vehicle severity  
- Pedestrian crossing and junction control impact

In [90]:
# ── 9a. Rural + High Speed (60/70 mph) ────────────────────────────────────────
rural_highspeed = df[(df['Area_Type'] == 'Rural') & (df['Speed_limit'].isin([60, 70]))]
print(f'Total rural high-speed accidents : {len(rural_highspeed):,}')
print(f'Fatal count                      : {(rural_highspeed["Accident_Severity"] == 1).sum():,}')
print(f'Fatal rate                       : {(rural_highspeed["Accident_Severity"] == 1).mean() * 100:.2f}%')
print('\nLight conditions breakdown:')
print(rural_highspeed.groupby('Light_Simple')['Is_Fatal'].agg(['mean', 'count']))

Total rural high-speed accidents : 170,195
Fatal count                      : 4,838
Fatal rate                       : 2.84%

Light conditions breakdown:
                   mean   count
Light_Simple                   
Dark-lit       0.027929   12711
Dark-no light  0.044500   32944
Dark-unknown   0.022099    1086
Dark-unlit     0.029654     607
Daylight       0.024217  122847


In [91]:
# ── 9b. Night (0–6h) + High Speed fatal profile ───────────────────────────────
night_highspeed = df[(df['Hour'] <= 6) & (df['Speed_limit'] >= 60)]
print(f'Total  : {len(night_highspeed):,}')
print(f'Fatal rate: {(night_highspeed["Accident_Severity"] == 1).mean() * 100:.2f}%')

Total  : 17,190
Fatal rate: 5.11%


In [92]:
# ── 9c. Single carriageway + Rural ────────────────────────────────────────────
# Single-carriageway rural roads lack central reservation / crash barriers
sc_rural = df[(df['Road_Type_Label'] == 'Single carriageway') & (df['Area_Type'] == 'Rural')]
print(f'Fatal rate     : {(sc_rural["Accident_Severity"] == 1).mean() * 100:.2f}%')
print(f'Avg speed limit: {sc_rural["Speed_limit"].mean():.1f} mph')

Fatal rate     : 2.42%
Avg speed limit: 48.6 mph


In [93]:
# ── 9d. Fatal rate trend: Urban vs Rural by Year ──────────────────────────────
year_area_fatal = df.groupby(['Year', 'Area_Type'])['Is_Fatal'].agg(['mean', 'count']).reset_index()
year_area_fatal['fatal_rate'] = (year_area_fatal['mean'] * 100).round(2)
pivot_ya = year_area_fatal.pivot(index='Year', columns='Area_Type', values='fatal_rate')
print(pivot_ya)

fig, ax = plt.subplots(figsize=(12, 5))
for col in ['Urban', 'Rural']:
    if col in pivot_ya.columns:
        ax.plot(pivot_ya.index, pivot_ya[col], marker='o', linewidth=2, label=col)
ax.set_title('Fatal Rate Trend: Urban vs Rural (by Year)', fontsize=14, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Fatal Rate (%)')
ax.legend()
save(fig, '27_fatal_rate_trend_urban_rural')

Area_Type  Rural  Unallocated  Urban
Year                                
2005        2.50          0.0   0.80
2006        2.50          0.0   0.91
2007        2.48          0.0   0.83
2009        2.20          NaN   0.74
2010        1.95          NaN   0.58
2011        2.04          NaN   0.66
2012        1.93          NaN   0.65
2013        2.04          NaN   0.61
2014        2.01          NaN   0.65


In [94]:
# ── 9e. Single vs Multi-vehicle severity ──────────────────────────────────────
df['Vehicle_Count_Type'] = df['Number_of_Vehicles'].apply(lambda x: 'Single' if x == 1 else 'Multi')
print(df.groupby('Vehicle_Count_Type')['Is_Fatal'].agg(['mean', 'count']))

                        mean   count
Vehicle_Count_Type                  
Multi               0.009043  648160
Single              0.018322  279227


In [95]:
# ── 9f. Pedestrian crossing × Severity ────────────────────────────────────────
df['Ped_Crossing'] = df['Pedestrian_Crossing-Physical_Facilities'].fillna('Unknown')
ped_sev = df.groupby('Ped_Crossing')['Is_Fatal'].agg(['mean', 'count']).reset_index()
ped_sev['fatal_rate'] = (ped_sev['mean'] * 100).round(2)
ped_sev = ped_sev[ped_sev['count'] > 100]
print(ped_sev.sort_values('fatal_rate', ascending=False))

# ── 9g. Junction control × Severity ──────────────────────────────────────────
df['Junction_Label'] = df['Junction_Control'].fillna('Unknown')
junc_sev = df.groupby('Junction_Label')['Is_Fatal'].agg(['mean', 'count']).reset_index()
junc_sev['fatal_rate'] = (junc_sev['mean'] * 100).round(2)
print(junc_sev.sort_values('fatal_rate', ascending=False))

                                  Ped_Crossing      mean   count  fatal_rate
1                         Footbridge or subway  0.017544    2622        1.75
0                               Central refuge  0.015408   15252        1.54
2        No physical crossing within 50 meters  0.012560  758173        1.26
6             non-junction pedestrian crossing  0.010258   49521        1.03
3  Pedestrian phase at traffic signal junction  0.006806   72727        0.68
5                               Zebra crossing  0.005846   29082        0.58

             Junction_Label      mean   count  fatal_rate
4                   Unknown  0.019152  351501        1.92
2   Giveway or uncontrolled  0.007661  462056        0.77
3                 Stop Sign  0.007310    4925        0.73
1  Automatic traffic signal  0.006156  107539        0.62
0         Authorised person  0.005124    1366        0.51


---
## Step 10 — Final Summary: Headline KPIs

In [96]:
total      = len(df)
fatals     = (df['Accident_Severity'] == 1).sum()
serious    = (df['Accident_Severity'] == 2).sum()
slight     = (df['Accident_Severity'] == 3).sum()
total_cas  = df['Number_of_Casualties'].sum()

print(f'\nTotal accidents : {total:,}')
print(f'Fatal           : {fatals:,} ({fatals/total*100:.2f}%)')
print(f'Serious         : {serious:,} ({serious/total*100:.2f}%)')
print(f'Slight          : {slight:,} ({slight/total*100:.2f}%)')
print(f'Total casualties: {int(total_cas):,}')
print(f'Avg casualties/accident: {total_cas/total:.2f}')

print(f'\nFatal rate rural : {df[df["Area_Type"]=="Rural"]["Is_Fatal"].mean()*100:.2f}%')
print(f'Fatal rate urban : {df[df["Area_Type"]=="Urban"]["Is_Fatal"].mean()*100:.2f}%')

print(f'\nFatal rate at 70 mph: {df[df["Speed_limit"]==70]["Is_Fatal"].mean()*100:.2f}%')
print(f'Fatal rate at 30 mph: {df[df["Speed_limit"]==30]["Is_Fatal"].mean()*100:.2f}%')

print(f'\nFatal rate in darkness (unlit): {df[df["Light_Simple"]=="Dark-unlit"]["Is_Fatal"].mean()*100:.2f}%')
print(f'Fatal rate in daylight        : {df[df["Light_Simple"]=="Daylight"]["Is_Fatal"].mean()*100:.2f}%')

print(f'\nPeak fatal hour   : {hour_fatal_rate["fatal_rate"].idxmax():.0f}:00')
print(f'Peak accident hour: {hour_counts.idxmax():.0f}:00')

print(f'\nAll charts saved to: {OUTPUT}')
print('\nANALYSIS COMPLETE.')

Total accidents : 927,387
Fatal           : 10,977 (1.18%)
Serious         : 122,236 (13.18%)
Slight          : 794,174 (85.64%)
Total casualties: 1,245,953
Avg casualties/accident: 1.34

Fatal rate rural : 2.22%
Fatal rate urban : 0.73%

Fatal rate at 70 mph: 2.32%
Fatal rate at 30 mph: 0.67%

Fatal rate in darkness (unlit): 1.74%
Fatal rate in daylight        : 0.95%

Peak fatal hour   : 4:00
Peak accident hour: 17:00

All charts saved to: ./UK_Accident_Analyzed/

ANALYSIS COMPLETE.
